In [2]:
import time
import numpy as np
import pandas as pd

from tqdm import tqdm
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from dataclasses import dataclass
from sklearn.metrics import average_precision_score

from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data

In [3]:
MODEL_DIR = Path.home() / "models" / "Qwen3-1.7B"
DATA_PATH = '../data/train.csv'
BENCHMARK_PATH = '../data/knowledge_bench_public.csv'
BENCHMARK_PATH = '../data/knowledge_bench_public.csv'

RANDOM_SEED = 42

df_train = pd.read_csv(DATA_PATH)
df_test = pd.read_csv(BENCHMARK_PATH)

In [30]:
import os
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer


os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

model_name = "Qwen/Qwen3-1.7B"


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",           
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

model.eval()  

#model params
device = next(model.parameters()).device
HIDDEN_SIZE = model.config.hidden_size
NUM_LAYERS = len(model.model.layers)

PROBE_LAYERS = [0, 5, 10, 15, 20, 25]
PROBE_LAYERS = [l for l in PROBE_LAYERS if l < NUM_LAYERS]

print(f"Model on: {device}")
print(f"Hidden size: {HIDDEN_SIZE}")
print(f"Number of layers: {NUM_LAYERS}")
print(f"Probe layers: {PROBE_LAYERS}")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:01<00:00, 240.92it/s]


Model on: cuda:0
Hidden size: 2048
Number of layers: 28
Probe layers: [0, 5, 10, 15, 20, 25]


In [10]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"  GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  GPU memory: 25.3 GB


In [12]:
UNCERTAINTY_FEATURES = 19  
INTERNAL_FEATURES_PER_LAYER = 6  
INTERNAL_FEATURES = len(PROBE_LAYERS) * INTERNAL_FEATURES_PER_LAYER + 2  

class FeatureAccumulator:

    def __init__(self, model, probe_layers: list[int] = PROBE_LAYERS):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks: list[torch.utils.hooks.RemovableHook] = []
        self._hidden: dict[str, torch.Tensor] = {}

    def attach(self):
        self._hidden.clear()
        for idx in self.probe_layers:
            name = f"layer_{idx}"

            def _make(n):
                def _fn(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()
                return _fn

            self._hooks.append(
                self.model.model.layers[idx].register_forward_hook(_make(name))
            )

    def detach(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def __enter__(self):
        self.attach()
        return self

    def __exit__(self, *_):
        self.detach()

    def compute_features(
        self,
        logits: torch.Tensor,
        input_ids: torch.Tensor,
        answer_start: int,
    ) -> dict:
        seq_len = input_ids.shape[1]
        n_answer = seq_len - answer_start

        if n_answer == 0:
            return {
                "uncertainty": np.zeros(UNCERTAINTY_FEATURES, dtype=np.float32),
                "internal_scalars": np.zeros(INTERNAL_FEATURES, dtype=np.float32),
                "probe_vec": np.zeros(HIDDEN_SIZE, dtype=np.float32),
            }
        answer_logits = logits[0, answer_start - 1: seq_len - 1, :].float()
        answer_ids = input_ids[0, answer_start:seq_len]
        log_probs = torch.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)

        probs = torch.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)
        
        probs_sorted, _ = probs.sort(dim=-1, descending=True)
        if probs_sorted.shape[-1] >= 2:
            top1_top2 = (probs_sorted[:, 0] - probs_sorted[:, 1]).mean().item() # new: разрыв между top1 и top2 вероятностями
        else:
            top1_top2 = 0.0

    
       
        lp_mean = token_lp.mean().item()
        lp_std = token_lp.std().item() if n_answer > 1 else 0.0
        cv = lp_std / (abs(lp_mean) + 1e-6)
        
        
        entropy_seq = entropy.cpu().numpy()
        if len(entropy_seq) > 1:
            slope = np.polyfit(range(len(entropy_seq)), entropy_seq, 1)[0]
        else:
            slope = 0.0
            
      
        if len(entropy_seq) > 1:
            p90 = np.percentile(entropy_seq, 90)
            n_spikes = (entropy_seq > p90).sum()
        else:
            n_spikes = 0

        uncertainty_arr = np.array([
            token_lp.mean().item(),
            token_lp.min().item(),
            token_lp.max().item(),
            token_lp.std().item() if n_answer > 1 else 0.0,
            entropy.mean().item(),
            entropy.max().item(),
            entropy.std().item() if n_answer > 1 else 0.0,
            torch.exp(-token_lp.mean()).item(),
            float(n_answer),
            token_lp[0].item(),
            top1.mean().item(),
            top5.mean().item(),
            top1_top2,
            cv,               
            slope,                 
            float(n_spikes),     
            entropy_seq[0] if len(entropy_seq) > 0 else 0.0,  
            entropy_seq[-1] if len(entropy_seq) > 0 else 0.0, 
            (entropy_seq[-1] - entropy_seq[0]) if len(entropy_seq) > 1 else 0.0,  # изменение энтропии
        ], dtype=np.float32)

        # внутри призн
        last_hs = self._hidden[f"layer_{self.probe_layers[-1]}"][0]
        probe_vector = last_hs[answer_start - 1].cpu().float().numpy()

        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[answer_start - 1].norm().item())
            int_scalars.append(hs[answer_start: seq_len].norm(dim=-1).mean().item())
            
            # STD норм во время ответа
            if n_answer > 1:
                int_scalars.append(hs[answer_start: seq_len].norm(dim=-1).std().item())
            else:
                int_scalars.append(0.0)

            # Logit Lens
            ans_hs = hs[answer_start - 1: seq_len - 1].unsqueeze(0)
            with torch.no_grad():
                ll = self.model.lm_head(self.model.model.norm(ans_hs)).float()
            ll_p = torch.softmax(ll[0], dim=-1)
            ll_e = -(ll_p * torch.log(ll_p + 1e-10)).sum(dim=-1)
            ll_max_prob = ll_p.max(dim=-1).values
            
            int_scalars.append(ll_e.mean().item())
            int_scalars.append(ll_e.std().item())  # STD энтропии Logit Lens
            int_scalars.append(ll_max_prob.mean().item())  # средняя max вероятность


        first_e = int_scalars[3]  # первая ll_e.mean 
        last_e = int_scalars[-3]  # последняя ll_e.mean
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e / (first_e + 1e-10))

        internal_scalar_arr = np.array(int_scalars, dtype=np.float32)

        self._hidden.clear()
        return {
            "uncertainty": uncertainty_arr,
            "internal_scalars": internal_scalar_arr,
            "probe_vec": probe_vector,
        }


def preprocess(
    tokenizer: AutoTokenizer,
    prompt: str,
    answer: str,
) -> tuple[torch.Tensor, int]:
  
    messages_prompt = [{"role": "user", "content": prompt}]
    prompt_enc = tokenizer.apply_chat_template(
        messages_prompt,
        add_generation_prompt=True,
        tokenize=True,
    )
    prompt_token_ids = prompt_enc["input_ids"]
    answer_start_idx = len(prompt_token_ids)

    messages_full = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    full_enc = tokenizer.apply_chat_template(messages_full, tokenize=True)
    token_ids = full_enc["input_ids"]
    token_ids = torch.tensor([token_ids], dtype=torch.long)
    
    return token_ids, answer_start_idx

## Добавленные признаки

1) Коэффициент вариации (cv)-std/mean log prob (показывает насколько неравномерна уверенность)
2) Тренд энтропии (slope)-наклон энтропии по токенам(галлюцинации часто имеют растущую энтропию)
3) Количество пиков-число токенов с энтропией >90%(резкие скачки неуверенности)
4) Энтропия первого/последнего токена и их разница(сравнение уверенности в начале и конце)
5) STD Logit Lens энтропи-разброс энтропии по позициям(+мера нестабильности)
6) Средняя max вероятность Logit Lens-насколько модель уверена в лучшем варианте (+сигнал уверенности)
7) STD норм скрытых состояний-разброс норм во время генерации-(аномалии в динамике)

In [14]:
accumulator = FeatureAccumulator(model)
unc_list, int_list, probe_list, label_list = [], [], [], []

for _, row in tqdm(df_train.iterrows(), total=len(df_train)): 
    token_ids, answer_start_idx = preprocess(tokenizer, row["prompt"], row["model_answer"])
    token_ids = token_ids.to(device)
   
    with accumulator:    
        with torch.no_grad():
            outputs = model(token_ids) 

    feats = accumulator.compute_features( 
        outputs.logits,
        token_ids,
        answer_start_idx
    
    )
    
    del outputs
   

    unc_list.append(feats["uncertainty"])
    int_list.append(feats["internal_scalars"])
    probe_list.append(feats["probe_vec"])
    label_list.append(row["is_hallucination"])
    

X_y_train = {
    "uncertainty_X":     np.stack(unc_list).astype(np.float32),
    "internal_scalar_X": np.stack(int_list).astype(np.float32),
    "probe_last_prompt": np.stack(probe_list).astype(np.float32),
    "labels":            np.array(label_list, dtype=np.int32),
} 

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1778/1778 [02:08<00:00, 13.80it/s]


In [16]:
import joblib
joblib.dump(X_y_train, 'X_y_train_qwen.joblib')

['X_y_train_qwen.joblib']

In [17]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import cross_val_score

class HallucinationClassifier:
    def __init__(self, clf, n_features=50,  use_feature_selection=True):
        
        self.selector = SelectKBest(score_func=mutual_info_classif, k=n_features) if use_feature_selection else None #отбор лучших признаков
        self.scaler = StandardScaler()
        
  
        self.clf = clf
        self.threshold = 0.5
        
     
        self.use_feature_selection = use_feature_selection


    def fit(self, X_y: dict):
        y = X_y["labels"]
        

        X = np.hstack([
            X_y["probe_last_prompt"],   
            X_y["internal_scalar_X"],
            X_y["uncertainty_X"],
        ])
        
        print(f"Исходная размерность: {X.shape[1]} признаков")
        

        X = self.scaler.fit_transform(X)
        
      
        if self.selector is not None:
            X = self.selector.fit_transform(X, y)
            print(f"После отбора признаков: {X.shape[1]} признаков")
        
      
        cv_scores = cross_val_score(self.clf, X, y, cv=3, scoring='average_precision')
        print(f"CV PR-AUC: {cv_scores.mean():.4f} (+- {cv_scores.std():.4f})")
        
        self.clf.fit(X, y)

    def to_X(self, features: dict) -> np.ndarray:
      
        X = np.hstack([
            features["probe_vec"].reshape(1, -1),
            features["internal_scalars"].reshape(1, -1),
            features["uncertainty"].reshape(1, -1),
        ])
        X = self.scaler.transform(X)
        
        if self.selector is not None:
            X = self.selector.transform(X)
        
        return X

    def predict_proba(self, features: dict) -> float:
        X = self.to_X(features)
        return float(self.clf.predict_proba(X)[0, 1])

    def predict(self, features: dict) -> tuple[int, float]:
        prob = self.predict_proba(features)
        return int(prob >= self.threshold), prob

подбор классификатора

In [23]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

models_to_try = {
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=5, random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.1,
        subsample=0.8, random_state=42, use_label_encoder=False,
        eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.1,
        subsample=0.8, random_state=42, verbose=-1
    ),
    'LogisticRegression': LogisticRegression(
        C=1.0, max_iter=3000, class_weight='balanced', random_state=42
    ),
}


trained_models = {}

for name, model in models_to_try.items():
    print(name)
    
    clf = HallucinationClassifier(clf=model, n_features=50, use_feature_selection=True)
    clf.fit(X_y_train)
    trained_models[name] = clf
    joblib.dump(clf, f"{name}.pkl")



print("SUMMARY: CV PR-AUC scores")

for name, clf in trained_models.items():
  
    print(f"{name}: trained")

GradientBoosting
Исходная размерность: 2105 признаков
После отбора признаков: 50 признаков
CV PR-AUC: 0.4264 (+- 0.0186)
RandomForest
Исходная размерность: 2105 признаков
После отбора признаков: 50 признаков
CV PR-AUC: 0.4093 (+- 0.0125)
XGBoost
Исходная размерность: 2105 признаков
После отбора признаков: 50 признаков
CV PR-AUC: 0.4253 (+- 0.0126)
LightGBM
Исходная размерность: 2105 признаков
После отбора признаков: 50 признаков
CV PR-AUC: 0.4336 (+- 0.0105)
LogisticRegression
Исходная размерность: 2105 признаков
После отбора признаков: 50 признаков
CV PR-AUC: 0.6045 (+- 0.0153)
SUMMARY: CV PR-AUC scores
GradientBoosting: trained
RandomForest: trained
XGBoost: trained
LightGBM: trained
LogisticRegression: trained


логистическая регрессия показала большой отрыв

In [27]:
@dataclass
class ScoringResult:
    is_hallucination: bool
    is_hallucination_proba: float
    t_model_sec: float = 0.0
    t_overhead_sec: float = 0.0
    t_total_sec: float = 0.0


class GuardianOfTruth:
    def __init__(self, model, tokenizer, classifier: HallucinationClassifier):
        self.model = model
        self.tokenizer = tokenizer
        self.classifier = classifier
        self.accumulator = FeatureAccumulator(model)

    def score(self, prompt: str, answer: str) -> ScoringResult:
        token_ids, answer_start_idx = preprocess(self.tokenizer, prompt, answer)
        device = next(self.model.parameters()).device
        token_ids = token_ids.to(device)

        t0 = time.perf_counter()
        with self.accumulator:
            with torch.no_grad():
                outputs = self.model(token_ids)
        t_model = time.perf_counter() - t0

    
        t1 = time.perf_counter()
        features = self.accumulator.compute_features(
            outputs.logits,
            token_ids,
            answer_start_idx,
        )
        del outputs
   

        if features is None:
            return ScoringResult(is_hallucination=False, is_hallucination_proba=0.0)

        label, prob = self.classifier.predict(features)
        t_overhead = time.perf_counter() - t1

        return ScoringResult(
            is_hallucination=bool(label),
            is_hallucination_proba=prob,
            t_model_sec=t_model,
            t_overhead_sec=t_overhead,
            t_total_sec=t_model + t_overhead,
        )

трейн на моем, оценка на публичном бенчмарке

In [ ]:
import joblib


clf = joblib.load("LogisticRegression.pkl")

guard = GuardianOfTruth(model, tokenizer, clf)


train_results = []
for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="train"):
    res = guard.score(row["prompt"], row["model_answer"])
    train_results.append(res)

train_probas = [r.is_hallucination_proba for r in train_results]  

test_results = []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="test"):
    res = guard.score(row["prompt"], row["model_answer"])
    test_results.append(res)

test_probas = [r.is_hallucination_proba for r in test_results]  

df_test_scored = df_test.copy()
df_test_scored["pred_proba"] = test_probas


print(f"Train PR-AUC: {average_precision_score(df_train['is_hallucination'], train_probas):.4f}")
print(f"Test PR-AUC (public benchmark): {average_precision_score(df_test['is_hallucination'], test_probas):.4f}")

test:  97%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉     | 1012/1044 [01:09<00:02, 15.18it/s]

Train PR-AUC: 0.6396
Test PR-AUC (public benchmark): 0.5412
